<a href="https://colab.research.google.com/github/Abhishek-Krishna-A-M/assistive-vision-ai/blob/main/ml_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
os.chdir('/content')

In [3]:
!rm -rf /content/assistive-vision-ai
!git clone https://github.com/Abhishek-Krishna-A-M/assistive-vision-ai /content/assistive-vision-ai
%cd /content/assistive-vision-ai

Cloning into '/content/assistive-vision-ai'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 34 (delta 16), reused 28 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 11.50 KiB | 203.00 KiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/assistive-vision-ai


In [4]:
!pip install uv
!uv pip install --system ultralytics transformers easyocr opencv-python pillow numpy gtts
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 86.7 MB/s eta 0:00:00
Using Python 3.13.15 environment at: /usr
Resolved 86 packages in 742ms
Prepared 8 packages in 116ms
Installed 8 packages in 12ms
 + easyocr==1.7.2
 + gtts==2.2.4
 + ninja==1.13.2
 + pyclipper==1.4.0
 + python-bidi==0.6.12
 + ultralytics==8.4.183
 + ultralytics-platform==0.1.90
 + ultralytics-thop==2.2.3


In [7]:
import sys
import os

repo_path = "/content/assistive-vision-ai"
if repo_path not in sys.path:
    sys.path.append(repo_path)

# Diagnostic check
video_processor_path = os.path.join(repo_path, 'video_processor.py')
if not os.path.exists(video_processor_path):
    print(f"Error: The file {video_processor_path} does not exist. Please ensure the repository was cloned correctly and the file is present.")
else:
    print(f"Found {video_processor_path}. Attempting import...")

import cv2
import base64
import numpy as np
from google.colab import output
from IPython.display import display, HTML

from video_processor import PipelineProcessor
from utils import annotate_frame

# Initialize model pipeline once
processor = PipelineProcessor(run_ocr=True)
frame_count = 0

def process_frame_callback(image_b64):
    global frame_count
    frame_count += 1

    # Decode frame
    header, data = image_b64.split(',')
    binary = base64.b64decode(data)
    img_array = np.frombuffer(binary, dtype=np.uint8)
    frame = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

    if frame is None:
        return ""

    # Inference (YOLO + Depth always, EasyOCR throttled to every 10th frame)
    detections = processor.detector.detect(frame)
    depth_map = processor.depth_estimator.estimate(frame)

    ocr_results = []
    if processor.ocr_reader and (frame_count % 10 == 0):
        ocr_results = processor.ocr_reader.read_text(frame)

    # Navigation & Speech
    instructions = processor.nav_engine.evaluate(detections, depth_map, ocr_results, frame.shape[1])
    if instructions:
        processor.speech.speak(instructions[0])

    # Annotate and downsample JPEG quality for fast bridge transfer
    annotated = annotate_frame(frame, detections, ocr_results, instructions)
    _, buffer = cv2.imencode('.jpg', annotated, [int(cv2.IMWRITE_JPEG_QUALITY), 50])
    encoded_out = base64.b64encode(buffer).decode('utf-8')

    return f"data:image/jpeg;base64,{encoded_out}"

output.register_callback('notebook.process_frame', process_frame_callback)

html_code = """
<div style="display: flex; gap: 10px;">
    <div>
        <h4>Live Feed</h4>
        <video id="webcam" playsinline style="width: 320px; height: 240px;" autoplay></video>
    </div>
    <div>
        <h4>AI Output</h4>
        <img id="processed" style="width: 320px; height: 240px; background: #222;" />
    </div>
</div>
<button id="stopBtn" style="margin-top: 10px;">Stop Stream</button>

<script>
(async function() {
    const video = document.getElementById('webcam');
    const img = document.getElementById('processed');
    const stopBtn = document.getElementById('stopBtn');

    const stream = await navigator.mediaDevices.getUserMedia({
        video: { facingMode: "environment", width: 320, height: 240 }
    });
    video.srcObject = stream;
    await video.play();

    const canvas = document.createElement('canvas');
    canvas.width = 320;
    canvas.height = 240;
    const ctx = canvas.getContext('2d');

    let running = true;
    let isProcessing = false; // Mutex lock to prevent Colab bridge backpressure

    stopBtn.onclick = () => {
        running = false;
        stream.getTracks().forEach(track => track.stop());
    };

    async function streamLoop() {
        if (!running) return;

        if (!isProcessing) {
            isProcessing = true;
            ctx.drawImage(video, 0, 0, canvas.width, canvas.height);
            const dataUrl = canvas.toDataURL('image/jpeg', 0.5);

            try {
                const result = await google.colab.kernel.invokeFunction(
                    'notebook.process_frame', [dataUrl], {}
                );
                if (result && result.data && result.data['text/plain']) {
                    const outputUrl = eval(result.data['text/plain']);
                    if (outputUrl) img.src = outputUrl;
                }
            } catch(e) {
                console.error(e);
            } finally {
                isProcessing = false;
            }
        }

        if (running) requestAnimationFrame(streamLoop);
    }

    streamLoop();
})();
</script>
"""

display(HTML(html_code))

Error: The file /content/video_processor.py does not exist. Please ensure the repository was cloned correctly and the file is present.
Initializing AI Pipeline Components...


Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

In [ ]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 7.2 MB/s eta 0:00:00
